# Healthcare Data Exploratory Analysis

This notebook demonstrates exploratory data analysis and predictive modeling for healthcare data.

## Objectives
1. Load and explore healthcare datasets
2. Visualize key patterns and relationships
3. Build predictive models for:
   - Disease outbreak prediction
   - Hospital admission forecasting
   - Patient outcome analysis

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# Add src directory to path
sys.path.append('../src')

from data_preprocessing import HealthDataPreprocessor, load_sample_health_data
from predictive_models import (
    DiseaseOutbreakPredictor,
    HospitalAdmissionPredictor,
    PatientOutcomePredictor,
    prepare_features_and_target
)
from visualization import (
    plot_feature_distributions,
    plot_correlation_matrix,
    plot_feature_importance,
    plot_confusion_matrix,
    plot_prediction_vs_actual
)

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Load and Explore Data

In [ ]:
# Load sample healthcare data
data = load_sample_health_data(n_samples=1000)
print(f"Dataset shape: {data.shape}")
data.head()

In [ ]:
# Display basic statistics
data.describe()

In [ ]:
# Check for missing values
missing_data = data.isnull().sum()
print("Missing values per column:")
print(missing_data[missing_data > 0])

## 2. Data Visualization

In [ ]:
# Plot feature distributions
numerical_cols = ['age', 'blood_pressure', 'heart_rate', 'temperature', 'respiratory_rate', 'length_of_stay']
plot_feature_distributions(data, numerical_cols)
plt.show()

In [ ]:
# Plot correlation matrix
plot_correlation_matrix(data)
plt.show()

In [ ]:
# Explore categorical variables
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

data['gender'].value_counts().plot(kind='bar', ax=axes[0], title='Gender Distribution')
data['admission_type'].value_counts().plot(kind='bar', ax=axes[1], title='Admission Type Distribution')
data['diagnosis_category'].value_counts().plot(kind='bar', ax=axes[2], title='Diagnosis Category Distribution')

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Initialize preprocessor
preprocessor = HealthDataPreprocessor()
preprocessor.data = data.copy()

# Handle missing values
data_processed = preprocessor.handle_missing_values(strategy='mean')

# Encode categorical variables
categorical_cols = ['gender', 'admission_type', 'diagnosis_category']
data_processed = preprocessor.encode_categorical(categorical_cols, method='onehot')

print(f"Processed data shape: {data_processed.shape}")
data_processed.head()

## 4. Predictive Modeling - Readmission Prediction

In [ ]:
# Prepare features and target
feature_cols = [col for col in data_processed.columns if col not in ['patient_id', 'readmission', 'mortality']]
X_train, X_test, y_train, y_test = prepare_features_and_target(
    data_processed, 'readmission', feature_cols
)

In [ ]:
# Train readmission prediction model
readmission_predictor = PatientOutcomePredictor(
    outcome_type='readmission',
    model_type='random_forest'
)
train_metrics = readmission_predictor.train(X_train, y_train)

In [ ]:
# Evaluate on test set
test_metrics = readmission_predictor.evaluate(X_test, y_test)
print("\nReadmission Prediction Results:")
print(f"Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall:    {test_metrics['recall']:.4f}")
print(f"F1-Score:  {test_metrics['f1_score']:.4f}")
print(f"ROC-AUC:   {test_metrics['roc_auc']:.4f}")

In [ ]:
# Plot confusion matrix
plot_confusion_matrix(
    test_metrics['confusion_matrix'],
    labels=['No Readmission', 'Readmission']
)
plt.show()

In [ ]:
# Plot feature importance
if readmission_predictor.feature_importance is not None:
    plot_feature_importance(readmission_predictor.feature_importance, top_n=10)
    plt.show()

## 5. Predictive Modeling - Length of Stay Prediction

In [ ]:
# Prepare features for length of stay prediction
los_feature_cols = [col for col in data_processed.columns 
                    if col not in ['patient_id', 'readmission', 'mortality', 'length_of_stay']]
X_train_los, X_test_los, y_train_los, y_test_los = prepare_features_and_target(
    data_processed, 'length_of_stay', los_feature_cols
)

In [ ]:
# Train length of stay prediction model
los_predictor = HospitalAdmissionPredictor(model_type='random_forest')
train_metrics_los = los_predictor.train(X_train_los, y_train_los)

In [ ]:
# Evaluate on test set
test_metrics_los = los_predictor.evaluate(X_test_los, y_test_los)
print("\nLength of Stay Prediction Results:")
print(f"R² Score: {test_metrics_los['r2']:.4f}")
print(f"RMSE:     {test_metrics_los['rmse']:.4f}")
print(f"MAE:      {test_metrics_los['mae']:.4f}")

In [ ]:
# Plot predictions vs actual
y_pred_los = los_predictor.predict(X_test_los)
plot_prediction_vs_actual(
    y_test_los.values,
    y_pred_los,
    title='Length of Stay: Predictions vs Actual'
)
plt.show()

## 6. Model Comparison and Insights

In [ ]:
# Compare different models for readmission prediction
models = ['random_forest', 'gradient_boosting', 'logistic']
results = {}

for model_type in models:
    predictor = PatientOutcomePredictor(
        outcome_type='readmission',
        model_type=model_type
    )
    predictor.train(X_train, y_train)
    metrics = predictor.evaluate(X_test, y_test)
    results[model_type] = metrics

# Display comparison
comparison_df = pd.DataFrame({
    model: {
        'Accuracy': metrics['accuracy'],
        'F1-Score': metrics['f1_score'],
        'ROC-AUC': metrics['roc_auc']
    }
    for model, metrics in results.items()
}).T

print("Model Comparison:")
print(comparison_df)

# Visualize comparison
comparison_df.plot(kind='bar', figsize=(12, 6))
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Conclusions

This notebook demonstrated:
1. Loading and preprocessing healthcare data
2. Exploratory data analysis with visualizations
3. Building predictive models for:
   - Patient readmission prediction
   - Length of stay forecasting
4. Model evaluation and comparison

### Key Findings:
- Multiple machine learning models can effectively predict healthcare outcomes
- Feature importance analysis helps identify key risk factors
- Model performance can be optimized through hyperparameter tuning

### Next Steps:
- Apply these models to real-world healthcare datasets
- Implement time-series analysis for disease outbreak prediction
- Deploy models for real-time prediction systems